# Identifying DRVI factors with LLM tools

LLM-based annotators take a factor's top marker genes and return a cell-type or
biological-process label in natural language. They need **no reference atlas and no
significance threshold**, so they can label factors the annotation- and enrichment-based tools
leave empty — which makes them useful **when neither user annotations nor a tissue-matched
CellTypist model are available**. This notebook covers:

1. **Direct LLM annotation** — a single, well-structured prompt you control, runnable against any
   backend (**Ollama**, **Claude API**, **Claude Code** (no API key), **OpenAI**, or **Gemini**).
   You own the prompt and can inspect exactly what the model was asked.
2. **CASSIA** — a multi-agent annotator (chain-of-thought → validation loop → structured output)
   with a quality score.
3. **gs2txt** — free-text process summaries that run pathway enrichment first, then one LLM call.

> **Why no AnnDictionary?** We previously included [AnnDictionary](https://github.com/ggit12/anndictionary).
> Its `ai_cell_type` / `ai_biological_process` are a single hardcoded one-line, zero-shot prompt
> (no reasoning, no validation), and the package pins `numpy`, `scanpy`, and `anndata` to old
> exact versions that conflict with a modern single-cell stack. The **Direct LLM** section below
> replaces it with a stronger, transparent prompt you can adapt — so we just mention it here.

> This is one of four companion notebooks. See
> [cell types from annotations](./identification_of_factors_1_cell_types.html),
> [biological processes via enrichment](./identification_of_factors_2_biological_processes.html),
> and [factor curation](./identification_of_factors_4_curation.html).
> All share `embed.h5ad`; the curation notebook picks up the results stored here.

> **⚠️ This notebook is not executed in CI, and its dependencies are intentionally left out of
> `requirements.txt`.** LLM output is fluent but produced *without an uncertainty signal* and can
> be confidently wrong, so always cross-check it against the SMI and enrichment tools and against
> the literature. Install the packages for your chosen backend via the install cell below.

## Prerequisites

Assumes a trained DRVI model with interpretability scores (see the
[general pipeline](./general_pipeline.html)).

**Adapting to your own model** — change `io_dir` (Section 0), pick `LLM_BACKEND`, set that
backend's model and credentials, and set `llm_tissue_context` / `llm_species`.

## Contact

Questions: [scverse discourse](https://discourse.scverse.org/). Bugs:
[issue tracker](https://github.com/theislab/drvi/issues).

## Install

Install only what your chosen backend needs (none of these are in `requirements.txt`):

- **Ollama** or **OpenAI**: `pip install openai`
- **Claude API**: `pip install anthropic`
- **Claude Code** (no API key — uses your Claude Code login): `pip install claude-agent-sdk`;
  also needs the `claude` CLI installed and logged in (`claude login`), and in Jupyter
  `pip install nest-asyncio`.
- **Gemini**: `pip install google-genai`
- **CASSIA** (Section 2): `pip install CASSIA`
- **gs2txt** (Section 3): `pip install "gs2txt[enrichment]"`

In [1]:
import sys
import subprocess

# Uncomment the line(s) for the backend/tools you want to use:
# subprocess.check_call([sys.executable, "-m", "pip", "install", "openai"])            # Ollama / OpenAI
# subprocess.check_call([sys.executable, "-m", "pip", "install", "anthropic"])         # Claude API
# subprocess.check_call([sys.executable, "-m", "pip", "install", "claude-agent-sdk", "nest-asyncio"])  # Claude Code
# subprocess.check_call([sys.executable, "-m", "pip", "install", "google-genai"])      # Gemini
# subprocess.check_call([sys.executable, "-m", "pip", "install", "CASSIA"])            # Section 2
# subprocess.check_call([sys.executable, "-m", "pip", "install", "gs2txt[enrichment]"])  # Section 3

## Imports

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
import json
import re
import asyncio
import pandas as pd
from pathlib import Path

import scvi
import drvi
from drvi.model import DRVI
import scanpy as sc

## 0. Setup

### Config

In [4]:
# Input/output directory holding the trained model and embeddings. Update accordingly.
io_dir = Path("./tmp_io/drvi_immune_128/").resolve()

# DRVI provides two complementary per-gene score matrices (both precomputed by the general pipeline):
#   OOD ("OOD_combined")             — SPECIFIC genes: highlights genes that uniquely mark a program;
#                                      genes shared across many programs are penalized.
#   IND ("IND_linear_weighted_mean") — DIRECT effect: the latent factor's effect on each gene, similar
#                                      to a log fold-change, so it also keeps differential genes that
#                                      are SHARED between programs.
score_key = "OOD_combined"                   # OOD (specific) — also used by CASSIA and gs2txt below
score_key_ind = "IND_linear_weighted_mean"   # IND (direct effect, logFC-like)

# Top genes sent to the LLM per score type, plus a cutoff for each score.
llm_top_n_genes = 100
drvi_score_cutoff = 0.5     # OOD cutoff (specific genes)
ind_score_cutoff = 0.5      # IND cutoff (direct-effect genes)

# Biological context passed to every tool.
llm_tissue_context = "human immune cells (PBMC / bone marrow)"
llm_species = "human"  # or "mouse"

# How many informative factor-directions to annotate. Set to an int for a quick/cheap smoke test
# (annotates the first N); None = annotate all. Applies to every section below.
# NOTE: the example outputs saved in this notebook were produced with the 8-direction sample below.
max_directions = 8

### Load model and embeddings

In [5]:
adata = sc.read_h5ad(io_dir / "adata_preprocesses.h5ad")
model = DRVI.load(io_dir / "drvi_model", adata)

embed_path = io_dir / "embed.h5ad"
embed = sc.read_h5ad(embed_path)

# Per-gene score matrices. scores_df (OOD, specific) is used by every tool; the direct-LLM section
# below also uses ind_scores (IND, direct effect) so the model sees both specific and shared genes.
scores_df = model.get_interpretability_scores(embed, adata, key=score_key)
ind_scores = model.get_interpretability_scores(embed, adata, key=score_key_ind)

INFO     File /Users/amir/projects/drvi_tutorials/tmp_io/drvi_immune_128/drvi_model/model.pt already downloaded    


INFO     DRVI: The model is trained with DRVI version 0.2.5.                                                       


INFO     DRVI: Updaging data setup config ...                                                                      


INFO     DRVI: Done updating data source registry. Loading in DRVI version 0.2.6.                                  


INFO     DRVI: Loading model from DRVI version 0.2.5.                                                              


INFO     DRVI: Done updating model args. Loading in 0.2.6.                                                         


INFO     DRVI: The model has been initialized                                                                      


## 1. Direct LLM annotation

Instead of relying on a wrapper package's hidden prompt, we send our **own** structured prompt
and choose the backend. For each factor-direction the model receives **both** DRVI score views —
the OOD *specific* genes and the IND *direct-effect* genes — with an explanation of what each
means, the tissue context, is asked to reason, and returns a small JSON object (`cell_type`,
`biological_process`, `key_genes`, `confidence`, `reasoning`) that we parse and store. Because you
control the prompt, you can adapt it to your tissue and see exactly what was asked.

### Choose a backend

Set `LLM_BACKEND` and fill in the model + credentials for that backend only:

- **`"ollama"`** — free, local/cluster, OpenAI-compatible. See the Ollama setup guide below.
- **`"claude"`** — Anthropic API via the `anthropic` SDK. Set `ANTHROPIC_API_KEY`.
- **`"claude_code"`** — Claude Agent SDK, which uses your existing **Claude Code login** — no API
  key needed. Requires the `claude` CLI installed and authenticated (`claude login`); the SDK
  talks to that local CLI process.
- **`"openai"`** — OpenAI API. Set `OPENAI_API_KEY`.
- **`"gemini"`** — Google Gemini API. Set `GEMINI_API_KEY` (or `GOOGLE_API_KEY`).

In [6]:
LLM_BACKEND = "claude_code"  # one of: "ollama", "claude", "claude_code", "openai", "gemini"

# Ollama (OpenAI-compatible; no API key needed)
OLLAMA_URL   = "http://supergpu22.scidom.de:8979"  # replace with your node and port
OLLAMA_MODEL = "qwen3.6:35b"

# Claude via the Anthropic API SDK — reads ANTHROPIC_API_KEY from the environment
CLAUDE_MODEL = "claude-opus-4-8"   # "claude-haiku-4-5" is cheaper/faster

# Claude via the Claude Agent SDK (uses your Claude Code login; no API key)
CLAUDE_CODE_MODEL = "opus"         # short alias ("opus"/"sonnet"/"haiku") or a full model ID

# OpenAI — reads OPENAI_API_KEY from the environment
OPENAI_MODEL = "gpt-4o"

# Gemini (google-genai) — reads GEMINI_API_KEY / GOOGLE_API_KEY from the environment
GEMINI_MODEL = "gemini-2.5-flash"

### Ollama setup guide (only if `LLM_BACKEND = "ollama"`)

[Ollama](https://ollama.com/) runs open LLMs (Llama 3, Qwen, ...) locally or on a cluster.

**0. Install** (one-time; clusters usually disallow Docker, so use Apptainer):
```bash
mkdir -p ~/containers
apptainer pull ~/containers/ollama.sif docker://ollama/ollama:latest
```
**1. Start a GPU job** via your scheduler (e.g. Slurm).
**2. Launch the container** (adjust bind paths):
```bash
apptainer shell --nv --bind /localscratch --bind /lustre/groups/ml01/ ~/containers/ollama.sif
```
**3. Start the server** on a custom port:
```bash
OLLAMA_HOST=0.0.0.0:8979 ollama serve &
```
**4. Pull a model**:
```bash
export OLLAMA_HOST=supergpu22.scidom.de/:8979
ollama pull qwen3.6:35b
```
Update `OLLAMA_URL` / `OLLAMA_MODEL` above to match.

### The backend dispatcher

One `call_llm(system, user)` function, five backends. Only the selected backend's package needs
to be installed.

In [7]:
def _run_async(coro):
    """Run an async coroutine from sync code, tolerating an already-running loop (e.g. Jupyter)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import nest_asyncio  # only needed inside a live loop (Jupyter)
    nest_asyncio.apply()
    return asyncio.get_event_loop().run_until_complete(coro)


def call_llm(system, user):
    if LLM_BACKEND == "ollama":
        from openai import OpenAI
        client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")
        resp = client.chat.completions.create(
            model=OLLAMA_MODEL,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0,
        )
        return resp.choices[0].message.content

    if LLM_BACKEND == "openai":
        from openai import OpenAI
        client = OpenAI()  # reads OPENAI_API_KEY
        resp = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            temperature=0,
        )
        return resp.choices[0].message.content

    if LLM_BACKEND == "claude":
        import anthropic
        client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY
        # Note: newer Claude models reject temperature/top_p — steer via the prompt instead.
        msg = client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=1024,
            system=system,
            messages=[{"role": "user", "content": user}],
        )
        return "".join(block.text for block in msg.content if block.type == "text")

    if LLM_BACKEND == "claude_code":
        # Claude Agent SDK — talks to your local, logged-in `claude` CLI (no API key).
        from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock

        async def _ask():
            text = ""
            options = ClaudeAgentOptions(
                system_prompt=system, model=CLAUDE_CODE_MODEL, max_turns=1, allowed_tools=[],
            )
            async for message in query(prompt=user, options=options):
                if isinstance(message, AssistantMessage):
                    for block in message.content:
                        if isinstance(block, TextBlock):
                            text += block.text
            return text

        return _run_async(_ask())

    if LLM_BACKEND == "gemini":
        from google import genai
        client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=f"{system}\n\n{user}")
        return resp.text

    raise ValueError(f"Unknown LLM_BACKEND: {LLM_BACKEND!r}")

### The predefined prompt

A fixed expert system prompt plus a per-factor user prompt that injects **two** ranked gene lists
(OOD = specific genes, IND = direct-effect / logFC-like genes), explains what each means, asks the
model to reason, and constrains the output to a small JSON object. Adapt the wording to your own
tissue/organism if needed.

In [8]:
IDENTIFY_SYSTEM = (
    "You are an expert computational biologist specializing in single-cell transcriptomics and "
    "immunology. You interpret latent gene programs learned by DRVI, a disentangled variational "
    "model. Each program is summarized by two complementary ranked marker-gene lists — a "
    "specificity score and a direct-effect score — whose meanings are explained in the prompt. "
    "Given these lists and the tissue context, identify what the program most likely represents, "
    "reasoning from established marker-gene biology. Be precise and do not overstate confidence "
    "when the genes are ambiguous."
)


def build_identify_prompt(factor_label, ood_genes, ind_genes, tissue):
    return (
        f"Tissue context: {tissue}\n"
        f"DRVI program: {factor_label}\n\n"
        "You are given two complementary ranked marker-gene lists for this program "
        "(both ranked most-influential first):\n\n"
        "1. SPECIFIC genes (OOD score): genes that most *specifically* mark this program. This "
        "score penalizes genes that are shared across many programs, so these are the program's "
        "most distinctive identity markers.\n"
        f"{', '.join(ood_genes)}\n\n"
        "2. DIRECT-EFFECT genes (IND score): the latent factor's direct effect on each gene, "
        "analogous to a log fold-change. It does NOT penalize sharing, so it also includes "
        "differential genes that are shared between programs — useful for reading the broader "
        "biological process and shared machinery.\n"
        f"{', '.join(ind_genes)}\n\n"
        "Use the SPECIFIC list mainly to pin down cell-type identity, and the DIRECT-EFFECT list to "
        "read the broader process (including shared genes). Reason from both, then give your answer "
        "as a JSON object with exactly these keys:\n"
        '  "cell_type": most likely cell type or cell state (or "unclear")\n'
        '  "biological_process": dominant biological process or pathway (or "unclear")\n'
        '  "key_genes": up to 5 genes that most support the call (list of strings)\n'
        '  "confidence": one of "high", "medium", "low"\n'
        '  "reasoning": one or two sentences justifying the call\n'
        "Respond with ONLY the JSON object — no surrounding text and no code fences."
    )

### Parse and run

LLMs sometimes wrap JSON in prose or code fences, so we extract the first JSON object defensively.

In [9]:
def parse_json(text):
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n?", "", text).rstrip("`").strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}


def top_genes(scores, col, cutoff, top_n):
    s = scores[col]
    return s[s >= cutoff].nlargest(top_n).index.astype(str).tolist()


def identify_factors(ood_scores, ind_scores, tissue, ood_cutoff, ind_cutoff, top_n, max_dirs=None):
    rows = []
    for col in ood_scores.columns:
        ood_genes = top_genes(ood_scores, col, ood_cutoff, top_n)
        if not ood_genes:  # uninformative direction — skip
            continue
        ind_genes = top_genes(ind_scores, col, ind_cutoff, top_n)
        parsed = parse_json(
            call_llm(IDENTIFY_SYSTEM, build_identify_prompt(col, ood_genes, ind_genes, tissue))
        )
        key_genes = parsed.get("key_genes")
        rows.append({
            "factor": col[:-1].strip(),
            "direction": col[-1],
            "cell_type": parsed.get("cell_type"),
            "biological_process": parsed.get("biological_process"),
            "key_genes": ", ".join(key_genes) if isinstance(key_genes, list) else key_genes,
            "confidence": parsed.get("confidence"),
            "reasoning": parsed.get("reasoning"),
        })
        print(f"{col}: {parsed.get('cell_type')} / {parsed.get('biological_process')}")
        if max_dirs is not None and len(rows) >= max_dirs:
            break
    return pd.DataFrame(rows)


llm_direct_results = identify_factors(
    scores_df, ind_scores, llm_tissue_context, drvi_score_cutoff, ind_score_cutoff,
    llm_top_n_genes, max_directions,
)
with pd.option_context("display.max_colwidth", None):
    display(llm_direct_results)

DR 1-: Naive/central-memory CD4+ T cell / Naive T-cell identity and quiescence/homing (TCR signaling, lymph-node homing, self-renewal)


DR 2-: Myeloid — classical monocyte / neutrophilic granulocyte / Innate immune / antimicrobial inflammatory response (calprotectin S100A8/A9/A12 alarmins, phagocytosis and pathogen sensing)


DR 3+: B cells (naive/immature B lymphocytes) / B-cell receptor signaling and B-cell identity/antigen presentation (MHC class II)


DR 4-: CD4+ effector/memory T helper cell (skin-homing Th2/Th22 phenotype) / T-helper effector differentiation and tissue/skin-homing chemokine programming


DR 5+: Cytotoxic CD56dim/CD16+ NK cells (mature/terminally differentiated effector) / NK-mediated cytotoxicity and granule-dependent killing


DR 6+: CD14+ classical monocytes / monocyte-macrophage lineage / Innate immune myeloid activation and inflammatory response (phagocytosis, antigen presentation, cytokine/chemokine signaling)


DR 7-: MAIT cells (mucosal-associated invariant T cells) / innate-like T cell cytotoxic/effector program (type-17 tissue-homing, granzyme-mediated killing)


DR 8-: CD8+ effector memory / terminally differentiated cytotoxic T cell (TEMRA), with NK-cell overlap / Cytotoxic effector function and terminal differentiation (granule-mediated killing)


,factor,direction,cell_type,biological_process,key_genes,confidence,reasoning
0,DR 1,-,Naive/central-memory CD4+ T cell,"Naive T-cell identity and quiescence/homing (TCR signaling, lymph-node homing, self-renewal)","CCR7, LEF1, TCF7, CD40LG, IL7R",high,"Pan-T identity genes (CD3D/E/G, CD3E, IL7R, CD2, CD5) combine with a strong naive/central-memory signature (CCR7, LEF1, TCF7, SELL-like TSHZ2/FHIT, MAL, SATB1, BACH2), and CD40LG plus the CD4-associated program point to naive/central-memory CD4+ T cells rather than a cytotoxic subset."
1,DR 2,-,Myeloid — classical monocyte / neutrophilic granulocyte,"Innate immune / antimicrobial inflammatory response (calprotectin S100A8/A9/A12 alarmins, phagocytosis and pathogen sensing)","S100A12, S100A8, S100A9, FCN1, CD14",high,"The specific list is dominated by calprotectin alarmins (S100A8/A9/A12) together with classical-monocyte markers (CD14, FCN1, VCAN, MNDA, LYZ) and granulocyte/neutrophil genes (RETN, RBP7, CSF3R, FPR1/2, PADI4), a hallmark inflammatory myeloid program; the direct-effect list reinforces this with antimicrobial and phagocytic machinery, though it cannot cleanly separate classical monocytes from neutrophilic granulocytes."
2,DR 3,+,B cells (naive/immature B lymphocytes),B-cell receptor signaling and B-cell identity/antigen presentation (MHC class II),"MS4A1, CD79A, CD19, TCL1A, PAX5",high,"The specific list is dominated by canonical B-cell identity and BCR-signaling genes (MS4A1/CD20, CD79A/B, CD19, CD22, PAX5, BLNK, BANK1, EBF1) with naive/immature markers TCL1A, FCER2/CD23, VPREB3, and CXCR5, and the direct-effect list adds BCR machinery (BTK, RASGRP3) plus MHC-II genes, together unambiguously marking a naive/immature B-cell program."
3,DR 4,-,CD4+ effector/memory T helper cell (skin-homing Th2/Th22 phenotype),T-helper effector differentiation and tissue/skin-homing chemokine programming,"CCR10, CCR4, CCR6, GATA3, TNFRSF4",medium,"The specific list (TNFRSF4/OX40, IL7R, IL32, LTB) marks memory CD4 T cells, and the direct-effect list is dominated by the CCR10+CCR4+CCR6 skin-homing chemokine axis plus GATA3, CD40LG, and FUT7, pointing to a Th2/Th22 effector-memory helper program; AIRE and Treg-associated genes (IL2RA, TNFRSF18) add some ambiguity, hence medium confidence."
4,DR 5,+,Cytotoxic CD56dim/CD16+ NK cells (mature/terminally differentiated effector),NK-mediated cytotoxicity and granule-dependent killing,"FGFBP2, KLRF1, NCR1, PRF1, GNLY",high,"The specific list is dominated by canonical NK identity markers (KLRF1/NKp80, NCR1/NKp46, KIR2DL3, KIR3DL2, CD160, SH2D1B/EAT-2, FCGR3A/CD16) together with cytotoxic effector genes (PRF1, GZMB/A/H/M, GNLY, NKG7, FASLG), while FGFBP2, S1PR5, CX3CR1 and B3GAT1/CD57 mark a terminally differentiated, highly cytotoxic CD56dim NK effector state rather than T cells (CD3-negative, lacking classic T-cell genes)."
5,DR 6,+,CD14+ classical monocytes / monocyte-macrophage lineage,"Innate immune myeloid activation and inflammatory response (phagocytosis, antigen presentation, cytokine/chemokine signaling)","CD14, FCN1, LYZ, S100A9, IL1B",high,"The specific list is dominated by canonical classical-monocyte/macrophage markers (CD14, FCN1, LYZ, S100A9, TYROBP, FCER1G, CD163, CYBB) with inflammatory effectors (IL1B, CCL3, FPR1), while the direct-effect list adds shared myeloid/APC machinery (CSF1R, CD68, VSIG4, MHC-II genes, CD1C/CLEC10A), consistent with an activated CD14+ monocyte program spanning phagocytic and antigen-presenting functions."
6,DR 7,-,MAIT cells (mucosal-associated invariant T cells),"innate-like T cell cytotoxic/effector program (type-17 tissue-homing, granzyme-mediated killing)","SLC4A10, KLRB1, IL23R, NCR3, GZMK",high,"The specific list is dominated by the canonical MAIT signature (SLC4A10, KLRB1/CD161, IL23R, NCR3, GZMK), and the direct-effect list adds ZBTB16/PLZF, CXCR6, CCR6, RORA, and IL18R1/IL18RAP together with CD3/CD8A and cytotoxic genes (PRF1, NKG7, GZMA/K/M), matching the type-17, tissue-homing, innate-like

### Store results

In [10]:
embed.uns["llm_direct_results"] = llm_direct_results.convert_dtypes(
    convert_integer=False, convert_floating=False
)
embed.var.set_index("title", drop=False, inplace=True)
for d, suf in [("+", "positive"), ("-", "negative")]:
    sub = llm_direct_results.query("direction == @d").set_index("factor")
    embed.var[f"{suf}_direction_llm_celltype"] = sub["cell_type"]
    embed.var[f"{suf}_direction_llm_process"] = sub["biological_process"]
embed.var.index = embed.var["original_dim_id"].astype(int).astype(str)
embed.var.index.name = None

**How to read this.** This runs on every factor, so it can give coarse orientation where SMI is
empty; when the gene list points clearly at one lineage, `cell_type` and `biological_process`
tend to agree and `confidence` is `"high"`. Because the output is fluent and self-reported, treat
`"low"`/`"medium"` confidence calls with caution and always cross-check against the SMI and
enrichment tools. The `key_genes` and `reasoning` fields let you trace each call back to the
factor's marker list.

## 2. CASSIA

[CASSIA](https://github.com/ElliotXie/CASSIA)
([Nature Comms 2025](https://www.nature.com/articles/s41467-025-67084-x)) is a multi-agent system:
a **chain-of-thought annotation agent**, a **validation agent** that loops (up to 3×) checking
marker consistency, and a **formatting agent** that emits a general + detailed cell type. Backends:
OpenAI, Anthropic, OpenRouter, or any OpenAI-compatible URL (Ollama). It writes CSV/JSON/HTML
reports to the working directory on each run (cleaned up below).

### Setup

In [11]:
import CASSIA

# CASSIA reaches an OpenAI-compatible endpoint; here we point it at Ollama.
cassia_output_name = "cassia_drvi"
cassia_provider = f"{OLLAMA_URL}/v1"
cassia_model = OLLAMA_MODEL

CASSIA.set_api_key("ollama", provider=cassia_provider)

### Run

In [12]:
def run_cassia_annotation(scores_df, tissue, cutoff, top_n, output_name, provider, model, species,
                          max_dirs=None):
    rows = []
    for col in scores_df.columns:
        genes = scores_df[col][scores_df[col] >= cutoff].nlargest(top_n).index.tolist()
        if genes:
            cluster_id = f"{col[:-1].strip().replace(' ', '_')}{col[-1]}"
            rows.append({"cluster": cluster_id, "gene": ", ".join(genes)})
            if max_dirs is not None and len(rows) >= max_dirs:
                break

    cassia_input = pd.DataFrame(rows)
    print(f"CASSIA input: {len(cassia_input)} factor-directions")

    CASSIA.runCASSIA_batch(
        marker=cassia_input, output_name=output_name, provider=provider, model=model,
        tissue=tissue, species=species, max_workers=4, validate_api_key_before_start=False,
    )

    results = pd.read_csv(f"{output_name}_summary.csv")
    results["factor"] = results["Cluster ID"].str[:-1].str.replace("_", " ")
    results["direction"] = results["Cluster ID"].str[-1]

    for p in Path(".").glob(f"{output_name}*"):
        p.unlink()
    return results


cassia_results = run_cassia_annotation(
    scores_df=scores_df, tissue=llm_tissue_context, cutoff=drvi_score_cutoff, top_n=llm_top_n_genes,
    output_name=cassia_output_name, provider=cassia_provider, model=cassia_model, species=llm_species,
    max_dirs=max_directions,
)
cassia_results.head()

CASSIA Batch Analysis ✓
[████████████████████████████████████████] 100%
Completed: 8 | Processing: 0 | Pending: 0
Active: None


  - DR_4-: LLM returned an empty response (provider=http://supergpu22.scidom.de:8979/v1, mo...
  - DR_2-: LLM returned an empty response (provider=http://supergpu22.scidom.de:8979/v1, mo...

All analyses completed. Results saved to 'cassia_drvi'.
HTML report generated: cassia_drvi_report.html
Three files have been created:
1. cassia_drvi_summary.csv (summary CSV)
2. cassia_drvi_conversations.json (conversation history JSON)
3. cassia_drvi_report.html (interactive HTML report)


,Cluster ID,Predicted General Cell Type,Predicted Detailed Cell Type,Possible Mixed Cell Types,Marker Number,Marker List,Iterations,Model,Provider,Tissue,Species,factor,direction
0,DR_1-,CD4+ T lymphocyte,"Naive CD4+ T Cell, Central Memory CD4+ T Cell ...",NaN,100,"TSHZ2, FHIT, CCR7, MDS2, EPHX2, CD40LG, AK5, L...",1,qwen3.6:35b,http://supergpu22.scidom.de:8979/v1,human immune cells (PBMC / bone marrow),human,DR 1,-
1,DR_3+,Germinal Center B Cell,"Germinal Center B Cell, Activated/Memory B Cel...","T Cell, NK Cell",100,"TCL1A, FCER2, MACROD2, MS4A1, CD72, VPREB3, FC...",1,qwen3.6:35b,http://supergpu22.scidom.de:8979/v1,human immune cells (PBMC / bone marrow),human,DR 3,+
2,DR_5+,Natural Killer (NK) Cell,"Mature Cytotoxic NK Cell (CD56dim/CD16bright),...",NK × epithelial/stromal,60,"FGFBP2, SPON2, MYOM2, PRF1, CCL4, NKG7, LAIR2,...",1,qwen3.6:35b,http://supergpu22.scidom.de:8979/v1,human immune cells (PBMC / bone marrow),human,DR 5,+
3,DR_6+,Classical Monocytes / Inflammatory Myeloid Cells,"Classical Monocytes / Inflammatory Monocytes, ...",NaN,35,"NRG1, CD14, FCN1, MARCO, LYZ, CLEC4E, CTSS, CC...",1,qwen3.6:35b,http://supergpu22.scidom.de:8979/v1,human immune cells (PBMC / bone marrow),human,DR 6,+
4,DR_7-,Natural Killer (NK) cell,"Mature Cytotoxic NK Cell (CD56^dim/CD16^+, KLR...",NaN,9,"SLC4A10, KLRB1, IL23R, NCR3, GZMK, KLRG1, MYBL...",1,qwen3.6:35b,http://supergpu22.scidom.de:8979/v1,human immune cells (PBMC / bone marrow),human,DR 7,-


### Store results

In [13]:
embed.uns["cassia_results"] = cassia_results.convert_dtypes(
    convert_integer=False, convert_floating=False
)
embed.var.set_index("title", drop=False, inplace=True)
for d, suf in [("+", "positive"), ("-", "negative")]:
    sub = cassia_results.query("direction == @d").set_index("factor")
    embed.var[f"{suf}_direction_cassia_general"] = sub["Predicted General Cell Type"]
    embed.var[f"{suf}_direction_cassia_detailed"] = sub["Predicted Detailed Cell Type"]
embed.var.index = embed.var["original_dim_id"].astype(int).astype(str)
embed.var.index.name = None

**How to read this.** The general + detailed tiers carry more information than a single label and
often reinforce each other on strong-signal factors; the general tier tends to be the more
reliable of the two. Treat it as coarse orientation and cross-check.

## 3. gs2txt

gs2txt runs pathway enrichment on the gene set first, then combines the enriched terms into a
structured prompt so the LLM produces a free-text process description. Providers: OpenAI,
Anthropic, or any OpenAI-compatible endpoint via `base_url` (Ollama). Install with the
`enrichment` extra so gseapy is available.

### Setup

In [14]:
from gs2txt import GeneSetAnnotator
from gs2txt.llm import OpenAIProvider

gs2txt_temperature = 0.1
gs2txt_enrichment_method = "pathway"

gs2txt_annotator = GeneSetAnnotator(
    llm_provider=OpenAIProvider(
        api_key="ollama", model_id=OLLAMA_MODEL,
        temperature=gs2txt_temperature, base_url=f"{OLLAMA_URL}/v1",
    ),
    enrichment_method=gs2txt_enrichment_method,
    organism=llm_species,
)

### Run

In [15]:
def run_gs2txt(scores_df, annotator, cutoff, top_n, context, max_dirs=None):
    rows = []
    for col in scores_df.columns:
        top = scores_df[col][scores_df[col] >= cutoff].nlargest(top_n)
        if top.empty:
            continue
        rows.append({
            "factor": col[:-1].strip(),
            "direction": col[-1],
            "description": annotator.annotate(
                pd.DataFrame({"gene": top.index, "logFC": top.values}),
                max_gene_num=top_n,
                additional_context=f"DRVI factor {col} - {context}",
            ),
        })
        if max_dirs is not None and len(rows) >= max_dirs:
            break
    return pd.DataFrame(rows)


gs2txt_results = run_gs2txt(
    scores_df, gs2txt_annotator, drvi_score_cutoff, llm_top_n_genes, llm_tissue_context, max_directions
)
with pd.option_context("display.max_colwidth", None):
    display(gs2txt_results)

,factor,direction,description
0,DR 1,-,"The perturbed gene set collectively orchestrates T cell receptor-mediated signaling and subsequent lymphocyte activation, coordinating downstream kinase cascades and transcriptional reprogramming to drive adaptive immune responses. This process encompasses early immunological synapse assembly, co-stimulatory checkpoint modulation, and lineage-specific differentiation toward effector subsets such as Th1 and Th2 cells. Dysregulation of these coordinated signaling events is central to adaptive immunity, immune tolerance, and T cell-driven inflammatory or autoimmune pathologies."
1,DR 2,-,"The perturbed genes and pathways collectively drive acute innate immune activation in myeloid cells, integrating pathogen recognition, complement cascade engagement, and phagocytic clearance mechanisms. This functional module coordinates pro-inflammatory signaling, reactive oxygen species production, and neutrophil extracellular trap formation to rapidly mount a cellular defense response against microbial invasion. The perturbation is primarily characterized by acute myeloid-mediated inflammatory response and antimicrobial effector function."
2,DR 3,+,"The perturbation predominantly orchestrates B cell receptor signaling and lymphocyte activation, driving clonal proliferation and differentiation within the adaptive immune system. Key components of the antigen recognition complex, co-stimulatory receptors, and lineage-specific transcription factors converge to initiate downstream signaling cascades that promote B cell maturation and effector function. This coordinated molecular program underpins the transition from naive B cells to activated humoral response states."
3,DR 4,-,"The perturbation drives the coordinated activation and clonal expansion of adaptive lymphocytes, primarily through T cell proliferation that subsequently supports B cell maturation and enhanced immunoglobulin secretion to amplify humoral and cellular immune responses. This reflects a broad upregulation of adaptive immune signaling and effector function."
4,DR 5,+,"The perturbed genes collectively encode cytotoxic granule effectors, activating and inhibitory receptors, and lineage-specifying transcription factors that drive natural killer cell activation and effector function. This molecular signature reflects the coordinated execution of target cell lysis, cytokine secretion, and immune surveillance, which underpins antiviral defense, tumor immunosurveillance, and alloimmune or autoimmune tissue clearance."
5,DR 6,+,This perturbation reflects innate immune pattern recognition coupled with growth factor-mediated myeloid cell survival and tissue repair signaling. Engagement of surface receptors initiates pathogen-sensing cascades that concurrently trigger ERBB2 and PI3K/AKT pathways to promote cellular proliferation and inhibit apoptosis. This molecular signature primarily characterizes a coordinated transition from innate immune activation toward pro-regenerative and survival-oriented myeloid responses.
6,DR 7,-,"The perturbation coordinately enhances natural killer cell activation, proliferation, and cytotoxic effector functions while promoting Th1 and Th17 lineage commitment and pro-inflammatory cytokine production. This immunomodulatory shift drives a coordinated innate and adaptive lymphocyte response characterized by increased leukocyte-mediated cytotoxicity and inflammatory polarization."
7,DR 8,-,"The perturbation orchestrates the differentiation and effector maturation of cytotoxic lymphocytes, integrating antigen receptor signaling with granzyme-mediated immune execution. This process coordinates CD8+ T cell lineage commitment, modulates growth factor-dependent proliferation, and fine-tunes type II interferon production to regulate adaptive immune responses. Consequently, it represents a core mechanism driving cytotoxic lymphocyte development and effector function activation."


### Store results

In [16]:
embed.uns["gs2txt_results"] = gs2txt_results.convert_dtypes(
    convert_integer=False, convert_floating=False
)
embed.var.set_index("title", drop=False, inplace=True)
for d, suf in [("+", "positive"), ("-", "negative")]:
    sub = gs2txt_results.query("direction == @d").set_index("factor")
    embed.var[f"{suf}_direction_gs2txt_label"] = sub["description"]
embed.var.index = embed.var["original_dim_id"].astype(int).astype(str)
embed.var.index.name = None

**How to read this.** Because gs2txt names genes and pathways, its summaries can be traced back to
the factor's top-ranked list — a useful gene-level complement to the ORA and TF tools. As with the
others, the output is fluent and unscored, so interpret it alongside the rest rather than on its own.

## 4. Save

In [17]:
import anndata as ad

ad.settings.allow_write_nullable_strings = True
embed.write_h5ad(embed_path)
print(f"Updated embedding saved to: {embed_path}")

... storing 'positive_direction_llm_celltype' as categorical


... storing 'positive_direction_llm_process' as categorical


... storing 'negative_direction_llm_celltype' as categorical


... storing 'negative_direction_llm_process' as categorical


... storing 'positive_direction_cassia_general' as categorical


...

... storing 'negative_direction_cassia_general' as categorical


... storing 'negative_direction_cassia_detailed' as categorical


... storing 'positive_direction_gs2txt_label' as categorical


... storing 'negative_direction_gs2txt_label' as categorical


Updated embedding saved to: /Users/amir/projects/drvi_tutorials/tmp_io/drvi_immune_128/embed.h5ad
